# 01 - Data Loading, Missingness Diagnostics & Kalman Imputation
**Market:** Agra | **Commodity:** Potato | **Period:** Jan 2010 - Jun 2025

This notebook:
1. Loads the raw Agmarknet export and reindexes to a full daily calendar.
2. Diagnoses the missingness pattern (gap lengths, % missing, seasonal clustering).
3. Empirically validates 3 imputation methods by masking known observations.
4. Applies the winning method to the real gaps and saves the clean series for
   downstream notebooks.

**Design choices & references** (see project README for full list):
- Kalman smoothing on a structural time series model (local trend + seasonal),
  per Moritz & Bartz-Beielstein (2017), Durbin & Koopman (2012).
- Compared against linear interpolation and STL-based seasonal-split imputation
  (Moritz et al. 2015; Cleveland et al. 1990) -- winner chosen empirically, not
  assumed.

## Colab setup
Run this once per session. It mounts Google Drive (where the whole `potato_pipeline/` project folder should be uploaded), `cd`s into `notebooks/` so every relative path (`../src`, `../data`, `../results`) keeps working unchanged, and installs the extra packages this notebook needs (Colab's base image doesn't include them).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Adjust this if you uploaded the project folder somewhere else in Drive
PROJECT_ROOT = "/content/drive/MyDrive/potato_pipeline"
os.chdir(os.path.join(PROJECT_ROOT, "notebooks"))
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q statsmodels

In [ ]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from imputation import (
    kalman_impute_structts,
    linear_interpolate,
    seasonal_decompose_interpolate,
    validate_imputation_methods,
    impute_positive_series,
)

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (11, 4)

## 1. Load raw data

Expected raw columns from Agmarknet exports typically look like:
`State, District, Market, Commodity, Variety, Grade, Arrival_Date, Min_Price, Max_Price, Modal_Price`

Adjust the column names below once the file is inspected. Place the Agra CSV at
`../data/agra_potato_raw.csv`.

In [ ]:
RAW_PATH = "../data/agra_potato_raw.xlsx"  # adjust filename/extension as needed

raw = pd.read_excel(RAW_PATH)  # use pd.read_csv(RAW_PATH) instead if your file is .csv
print(raw.shape)
raw.head()

In [ ]:
# --- Standardize columns: adjust names to match your actual file ---
DATE_COL = "Date"
PRICE_COL = "Price"

df = raw[[DATE_COL, PRICE_COL]].copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL], dayfirst=True, errors="coerce")
df = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL)

# If multiple records land on the same date, aggregate.
df = df.groupby(DATE_COL, as_index=True).agg({PRICE_COL: "mean"})
df.index.name = "date"
df.head()

## 2. Reindex to full daily calendar and inspect missingness

In [ ]:
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="D")
df_full = df.reindex(full_idx)
df_full.index.name = "date"

# Trim to the actual observed price range: if the earliest/latest calendar
# rows have a Date but a blank/NaN Price (common in raw Excel exports), the
# Kalman smoother has no data to extrapolate from before/after the series and
# can produce meaningless (even negative) edge values. There is also no
# principled way to impute a price before/after the data collection window,
# so we only ever impute BETWEEN two known points, never beyond them.
first_valid = df_full[PRICE_COL].first_valid_index()
last_valid = df_full[PRICE_COL].last_valid_index()
n_trimmed = len(df_full) - len(df_full.loc[first_valid:last_valid])
if n_trimmed > 0:
    print(f"Trimmed {n_trimmed} leading/trailing row(s) with no anchoring price observation.")
df_full = df_full.loc[first_valid:last_valid].copy()

n_total = len(df_full)
n_missing = df_full[PRICE_COL].isna().sum()
print(f"Total days: {n_total} | Missing: {n_missing} ({n_missing/n_total:.2%})")

In [ ]:
# Gap-length distribution (consecutive NaN runs)
is_na = df_full[PRICE_COL].isna().astype(int)
gap_id = (is_na.diff() == 1).cumsum() * is_na
gap_lengths = is_na.groupby(gap_id).sum()
gap_lengths = gap_lengths[gap_lengths > 0]

print(gap_lengths.describe())
gap_lengths.value_counts().sort_index().plot(kind="bar", title="Gap-length distribution (days)")
plt.xlabel("Gap length (consecutive missing days)")
plt.ylabel("Count of gaps")
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal clustering of missingness (e.g., market holidays, harvest lulls)
monthly_missing = df_full[PRICE_COL].isna().groupby(df_full.index.month).mean()
monthly_missing.plot(kind="bar", title="Fraction missing by calendar month")
plt.ylabel("Fraction missing")
plt.xlabel("Month")
plt.tight_layout()
plt.show()

## 3. Empirical validation of imputation methods

We mask a random 10% of currently-OBSERVED points, impute with each candidate
method, and compare reconstruction error against the known truth. This is done
on a relatively complete sub-segment to avoid compounding real gaps with
artificial ones.

In [ ]:
# Use the sub-segment with the fewest real gaps for a clean validation testbed
window = 730  # ~2 years
rolling_missing = df_full[PRICE_COL].isna().rolling(window).sum()
best_end = rolling_missing.idxmin()
best_start = best_end - pd.Timedelta(days=window - 1)
clean_segment = df_full.loc[best_start:best_end, PRICE_COL].copy()
print(f"Validation segment: {best_start.date()} to {best_end.date()}, "
      f"{clean_segment.isna().mean():.2%} missing")

# Fill any tiny residual gaps in the segment itself before masking (so 'truth' is known)
clean_segment = clean_segment.interpolate(method="time").ffill().bfill()

In [ ]:
# Wrap each method to impute in log-space, guaranteeing strictly positive
# imputed prices (see impute_positive_series docstring for rationale).
def linear_interp_pos(s, **kw):
    return impute_positive_series(s, linear_interpolate, **kw)

def kalman_pos(s, **kw):
    return impute_positive_series(s, kalman_impute_structts, **kw)

def stl_pos(s, **kw):
    return impute_positive_series(s, seasonal_decompose_interpolate, **kw)

methods = {
    "linear_interp": linear_interp_pos,
    "kalman_structts": kalman_pos,
    "stl_seasonal": stl_pos,
}
method_kwargs = {
    "kalman_structts": {"seasonal_period": 7, "trend": True},
    "stl_seasonal": {"period": 365, "robust": True},
}

validation_results = validate_imputation_methods(
    clean_segment, methods, mask_frac=0.10, n_trials=20, random_state=42, **method_kwargs
)
validation_results

In [ ]:
validation_results.set_index("method")[["RMSE_mean", "MAE_mean"]].plot(
    kind="bar", yerr=validation_results.set_index("method")[["RMSE_std", "MAE_std"]].values.T,
    capsize=4, title="Imputation method comparison (lower is better)"
)
plt.ylabel("Error (Rs/quintal)")
plt.tight_layout()
plt.show()

BEST_METHOD = validation_results.iloc[0]["method"]
print(f"Selected method for final imputation: {BEST_METHOD}")

## 4. Apply the winning method to the real gaps

By default this follows the paper-driven choice (Kalman on a structural model),
but the cell above empirically confirms/overrides it based on `BEST_METHOD`.

In [ ]:
if BEST_METHOD == "kalman_structts":
    imputed_price, fit_info = kalman_pos(df_full[PRICE_COL], seasonal_period=7, trend=True)
elif BEST_METHOD == "stl_seasonal":
    imputed_price, fit_info = stl_pos(df_full[PRICE_COL], period=365)
else:
    imputed_price, fit_info = linear_interp_pos(df_full[PRICE_COL])

df_full["price_imputed"] = imputed_price
# NOTE: no low/high columns in this dataset (Date, Price only).
# The PAE metric in Notebook 04, which needs daily high/low, will be skipped.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df_full.index, df_full["price_imputed"], label="Imputed", color="tab:orange", lw=1, zorder=1)
ax.plot(df_full.index, df_full[PRICE_COL], label="Observed", color="tab:blue", lw=1, zorder=2)
ax.set_title(f"Agra Potato Modal Price -- Observed vs Imputed ({BEST_METHOD})")
ax.set_ylabel("Rs/quintal")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Sanity checks before saving

In [ ]:
assert df_full["price_imputed"].isna().sum() == 0, "Imputation left residual NaNs"
assert (df_full["price_imputed"] > 0).all(), "Non-positive imputed prices detected -- inspect"
print("Sanity checks passed.")
print(df_full["price_imputed"].describe())

In [ ]:
out = df_full[["price_imputed"]].rename(columns={"price_imputed": "price"})
out.to_csv("../data/agra_potato_clean.csv")
print("Saved: ../data/agra_potato_clean.csv")
out.tail()